# 1 · Basic ReAct Agent
## How Agents Reason and Use Tools
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/1-basic-react-agent/basic_react_agent_workbook.ipynb)

Most LLM apps are one-shot: send a prompt, get a response, done.  
An **agent** is different — it loops. It can *reason* about a problem, *call a tool* to get new information, *observe* the result, then decide what to do next.

This is the **ReAct** pattern (Reason + Act), and it's the foundation of almost every production AI agent.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | Why agents? The limits of one-shot LLMs |
| 2 | The ReAct loop: reason → act → observe |
| 3 | Defining tools with `@tool` |
| 4 | Building a ReAct agent with LangGraph |
| 5 | Running the agent and reading the trace |
| ★ | Exercise: add your own tool |

---

### Prerequisites
- Python 3.10+
- `OPENAI_API_KEY` set in `.env` or Colab Secrets

### Key References
> Yao, S., et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629  
> LangGraph docs: https://langchain-ai.github.io/langgraph/  
> LangChain tools guide: https://python.langchain.com/docs/concepts/tools/

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "pydantic", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local environment — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded API key from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded API key from .env file.")

key = os.environ.get("OPENAI_API_KEY", "")
if key and key.startswith("sk-"):
    print("✓ OPENAI_API_KEY is set and looks valid.")
else:
    print("✗ OPENAI_API_KEY missing or invalid.")
    print("  Colab: Secrets panel (key icon) → Add secret 'OPENAI_API_KEY'")
    print("  Local: create a .env file with OPENAI_API_KEY=sk-...")

---

## Part 1 — Why Agents? The Problem With One-Shot LLMs

---

A standard LLM call looks like this:

```
You → "What is 247 multiplied by 13?"
LLM → "The answer is 3,211."   ← but is it right? The model guesses.
```

LLMs are trained to *predict likely tokens*, not to *compute*. For math, date logic, database lookups, or API calls, they hallucinate confident-sounding answers.

**The fix:** give the model *tools* it can actually run, and let it *choose* when to use them.

```
You  → "What is 247 multiplied by 13?"
Agent → [thinks] "I should use the multiply tool."
Agent → [calls]  multiply(247, 13) → 3211
Agent → [sees]   "The tool returned 3211."
Agent → "247 × 13 = 3,211."   ← grounded in real computation
```

This is the agent loop. The model never does the math itself — it orchestrates tools that do.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Without tools: model guesses
response = llm.invoke("What is 247 multiplied by 13? Just give the number.")
print("Without tools:", response.content)
print("Correct answer:", 247 * 13)
print()
print("The model may or may not be right — there is no guarantee.")

---

## Part 2 — The ReAct Loop

---

**ReAct** stands for **Re**asoning + **Act**ing. The loop has three steps:

```
┌─────────────────────────────────────────┐
│              ReAct Loop                 │
│                                         │
│   START                                 │
│     │                                   │
│     ▼                                   │
│  ┌──────────────────────┐               │
│  │   agent (LLM)        │               │
│  │                      │               │
│  │  "I need to multiply │               │
│  │   247 × 13.          │               │
│  │   I'll call multiply."│              │
│  └──────────┬───────────┘               │
│             │ tool_calls present?        │
│             ▼                           │
│  ┌──────────────────────┐               │
│  │   tools              │               │
│  │                      │               │
│  │  multiply(247, 13)   │               │
│  │  → 3211              │               │
│  └──────────┬───────────┘               │
│             │ result back to agent       │
│             ▼                           │
│  ┌──────────────────────┐               │
│  │   agent (LLM)        │               │
│  │                      │               │
│  │  "The tool returned  │               │
│  │   3211. I can answer."│              │
│  └──────────┬───────────┘               │
│             │ no more tool_calls         │
│             ▼                           │
│           END                           │
└─────────────────────────────────────────┘
```

The key insight: **the agent decides when to stop**. If the answer requires two tool calls, it loops twice. If one is enough, it stops after one.

---

## Part 3 — Defining Tools with `@tool`

---

A **tool** is any Python function decorated with `@tool`. LangChain reads the function name, type hints, and docstring to build a description the model can understand.

**The docstring is the instruction.** Write it as if you're telling the model what this function does and when to use it.

In [ ]:
from langchain_core.tools import tool


@tool
def add(x: int, y: int) -> int:
    """Add two integers and return their sum. Use when you need to add numbers."""
    return x + y


@tool
def multiply(x: int, y: int) -> int:
    """Multiply two integers and return their product. Use when you need to multiply numbers."""
    return x * y


# The model sees this description — it drives tool selection
print("Tool name:", multiply.name)
print()
print("Tool description:")
print(multiply.description)
print()
print("Tool schema (what the model sees):")
import json
print(json.dumps(multiply.args_schema.model_json_schema(), indent=2))

---

## Part 4 — Building the Agent

---

LangGraph's `create_react_agent` wires everything together:
1. Binds your tools to the LLM
2. Builds the reason → act → observe loop automatically
3. Returns a compiled graph you can stream or invoke

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

TOOLS = [add, multiply]

SYSTEM = SystemMessage(
    "You are a math assistant. Solve problems using only the provided tools. "
    "Never compute answers yourself — always use a tool."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(model=llm, tools=TOOLS, prompt=SYSTEM)

print("Agent compiled. Graph: START → agent ⇄ tools → END")

---

## Part 5 — Running the Agent and Reading the Trace

---

`agent.stream()` with `stream_mode="values"` shows every step of the loop as it happens — you can see the model's reasoning, each tool call, and the final answer.

In [ ]:
from langchain_core.messages import HumanMessage

question = "What is (3 + 4) multiplied by 5?"
print(f"Question: {question}")
print("-" * 50)

inputs = {"messages": [HumanMessage(question)]}

for step in agent.stream(inputs, stream_mode="values"):
    msg = step["messages"][-1]
    msg.pretty_print()

In [ ]:
# Try a harder problem — two tool calls needed
question2 = "Add 15 and 27, then multiply the result by 3."
print(f"Question: {question2}")
print("-" * 50)

inputs2 = {"messages": [HumanMessage(question2)]}

for step in agent.stream(inputs2, stream_mode="values"):
    msg = step["messages"][-1]
    msg.pretty_print()

---

## Exercise — Add Your Own Tool

Add a `subtract` tool to the agent and test it.

**Goal:** The agent should be able to solve: `"What is 100 minus 37?"`

**Steps:**
1. Define a `subtract(x, y)` tool with a clear docstring
2. Add it to `TOOLS`
3. Rebuild the agent with `create_react_agent`
4. Run the question and verify the agent calls `subtract`

In [ ]:
# ── Exercise Starter ────────────────────────────────────────────────────────

# TODO: define a subtract tool
# @tool
# def subtract(x: int, y: int) -> int:
#     """..."""
#     ...

# TODO: add subtract to TOOLS and rebuild the agent
# TOOLS_V2 = [add, multiply, subtract]
# agent_v2 = create_react_agent(...)

# TODO: run this question
# question3 = "What is 100 minus 37?"

In [ ]:
# ── Answer Key ──────────────────────────────────────────────────────────────

@tool
def subtract(x: int, y: int) -> int:
    """Subtract y from x and return the result. Use when you need to subtract numbers."""
    return x - y

TOOLS_V2 = [add, multiply, subtract]
agent_v2 = create_react_agent(model=llm, tools=TOOLS_V2, prompt=SYSTEM)

print("Question: What is 100 minus 37?")
print("-" * 50)
for step in agent_v2.stream({"messages": [HumanMessage("What is 100 minus 37?")]}, stream_mode="values"):
    step["messages"][-1].pretty_print()

---

## What's Next?

You now understand the ReAct loop — the core pattern behind most production agents. Here is where to go next:

- **Example 2 — Email Triage** ([`../2-email-triage/`](../2-email-triage/)): apply agents to a real business problem. Instead of math tools, the agent classifies emails into typed categories using Pydantic schemas.
- **Example 4 — Lead Qualifier** ([`../4-lead-qualifier/`](../4-lead-qualifier/)): add a scoring rubric to the prompt so the agent's decisions are explainable and auditable.

### Further Reading
- Yao et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models.* https://arxiv.org/abs/2210.03629
- LangGraph ReAct agent: https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/
- LangChain tools: https://python.langchain.com/docs/concepts/tools/

---
*Workshop complete. Next: Example 2 — Email Triage Agent.*